# Transkription aller Forster-Seiten

Modell: `checkpoints/best_cook_large/` (TrOCR Large fine-tuned on Cook)  
Eingabe: `data/all_manifests/manifest.json` → Zeilen-Crops aus `data/all_line_crops/`  
Ausgabe:  
- `data/transcriptions/<page_id>.txt` — Seitentranskription (resumable)  
- `data/raw_document.txt` — Gesamtdokument mit Buch- und Seitennummer

**923 Seiten / ~31 000 Zeilen** — ca. 30–60 min auf GPU.

In [1]:
import gc
import json
import re
import warnings
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

warnings.filterwarnings('ignore')

REPO_ROOT        = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

CHECKPOINT_DIR   = REPO_ROOT / 'checkpoints' / 'best_cook_large_v4'
MANIFEST_PATH    = REPO_ROOT / 'data' / 'all_manifests' / 'manifest.json'
TRANSCR_DIR      = REPO_ROOT / 'data' / 'transcriptions'
RAW_DOC_PATH     = REPO_ROOT / 'data' / 'raw_document.txt'
TRANSCR_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE       = 8
MAX_NEW_TOKENS   = 128
NUM_BEAMS        = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {device}')
print(f'Checkpoint : {CHECKPOINT_DIR}')
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(device).total_memory / 1024**3
    print(f'GPU memory : {total_gb:.1f} GB')

Device     : cuda
Checkpoint : /home/justin/Ginger_Gradient/14/project/Capstone-Project/checkpoints/best_cook_large_v4
GPU memory : 11.6 GB


In [2]:
gc.collect()
torch.cuda.empty_cache()

processor = TrOCRProcessor.from_pretrained(CHECKPOINT_DIR)
model     = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_DIR).to(device)
model.eval()

print(f'Modell geladen: {CHECKPOINT_DIR.name}')
print(f'Parameter     : {sum(p.numel() for p in model.parameters()):,}')

Loading weights:   0%|          | 0/636 [00:00<?, ?it/s]

Modell geladen: best_cook_large_v4
Parameter     : 558,226,432


In [3]:
with open(MANIFEST_PATH, encoding='utf-8') as f:
    global_manifest = json.load(f)

# Seiten nach Buch (B1→B6) und dann Seitennummer sortieren
def sort_key(page_entry):
    m = re.match(r'B(\d+)_P(\d+)', page_entry['page_id'])
    return (int(m.group(1)), int(m.group(2))) if m else (99, 99)

pages = sorted(global_manifest['pages'], key=sort_key)

already_done = sum(1 for p in pages
                   if (TRANSCR_DIR / f"{p['page_id']}.txt").exists())

print(f'Seiten gesamt        : {len(pages)}')
print(f'Zeilen gesamt        : {global_manifest["total_lines"]}')
print(f'Bereits transkribiert: {already_done}')
print(f'Verbleibend          : {len(pages) - already_done}')

Seiten gesamt        : 923
Zeilen gesamt        : 30959
Bereits transkribiert: 0
Verbleibend          : 923


In [4]:
class LineDataset(Dataset):
    """Lädt Zeilenbilder für eine Seite."""
    def __init__(self, image_paths, processor):
        self.paths     = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)


def transcribe_page(line_paths: list[Path]) -> list[str]:
    """Transkribiert alle Zeilenbilder einer Seite; gibt Liste von Strings zurück."""
    if not line_paths:
        return []

    dataset = LineDataset(line_paths, processor)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    texts   = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            ids   = model.generate(batch, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
            texts.extend(processor.batch_decode(ids, skip_special_tokens=True))

    return texts


def page_header(page_id: str) -> str:
    """Erzeugt einen Kopfzeilenstring: '=== Buch 1, Seite 012 ===' """
    m = re.match(r'B(\d+)_P(\d+)', page_id)
    if m:
        return f'=== Buch {m.group(1)}, Seite {m.group(2)} ==='
    return f'=== {page_id} ==='


print('Funktionen definiert.')

Funktionen definiert.


---
## Transkriptions-Pipeline

Bereits vorhandene `data/transcriptions/<page_id>.txt` werden übersprungen — der Lauf ist resumable.

In [5]:
for page_entry in tqdm(pages, desc='Seiten transkribieren'):
    page_id   = page_entry['page_id']
    out_path  = TRANSCR_DIR / f'{page_id}.txt'

    # Resume: überspringen wenn schon fertig
    if out_path.exists():
        continue

    # Seitenmanifest laden → Zeilen-Crop-Pfade in Reihenfolge
    page_manifest_path = REPO_ROOT / 'data' / 'all_manifests' / f'{page_id}.json'
    with open(page_manifest_path, encoding='utf-8') as f:
        page_manifest = json.load(f)

    line_paths = [
        REPO_ROOT / rec['line_image']
        for rec in page_manifest['lines']
        if (REPO_ROOT / rec['line_image']).exists()
    ]

    # Transkribieren
    texts = transcribe_page(line_paths)

    # Seitentranskription speichern
    header = page_header(page_id)
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(header + '\n')
        for text in texts:
            f.write(text + '\n')

    tqdm.write(f'{page_id}: {len(texts)} Zeilen')

print('\nTranskription abgeschlossen.')

Seiten transkribieren:   0%|          | 1/923 [00:01<29:03,  1.89s/it]

B1_P012: 13 Zeilen


Seiten transkribieren:   0%|          | 2/923 [00:06<57:49,  3.77s/it]

B1_P014: 30 Zeilen


Seiten transkribieren:   0%|          | 3/923 [00:11<1:04:56,  4.24s/it]

...

B6_P220: 36 Zeilen


Seiten transkribieren: 100%|█████████▉| 922/923 [1:56:53<00:07,  7.82s/it]

B6_P221: 34 Zeilen


Seiten transkribieren: 100%|██████████| 923/923 [1:56:53<00:00,  7.60s/it]

B6_P234: 4 Zeilen

Transkription abgeschlossen.


---
## raw_document.txt zusammenstellen

Alle Seiten in sortierter Reihenfolge (B1→B6, Seite aufsteigend) in eine einzige Textdatei.

In [6]:
total_lines_written = 0
missing_pages       = []

with open(RAW_DOC_PATH, 'w', encoding='utf-8') as out:
    for page_entry in pages:
        page_id  = page_entry['page_id']
        txt_path = TRANSCR_DIR / f'{page_id}.txt'

        if not txt_path.exists():
            missing_pages.append(page_id)
            continue

        content = txt_path.read_text(encoding='utf-8').strip()
        out.write(content + '\n\n')

        # Zeilen zählen (ohne den Header)
        lines = [l for l in content.splitlines() if not l.startswith('===')]
        total_lines_written += len(lines)

print(f'raw_document.txt geschrieben: {RAW_DOC_PATH}')
print(f'Seiten eingebunden  : {len(pages) - len(missing_pages)} / {len(pages)}')
print(f'Zeilen gesamt       : {total_lines_written}')
if missing_pages:
    print(f'Fehlende Seiten ({len(missing_pages)}): {missing_pages[:10]}...')

raw_document.txt geschrieben: /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/raw_document.txt
Seiten eingebunden  : 923 / 923
Zeilen gesamt       : 30957


In [7]:
# Überblick: Zeilen pro Buch + Vorschau erste Seite
from collections import defaultdict

book_lines = defaultdict(int)
book_pages = defaultdict(int)

for page_entry in pages:
    page_id  = page_entry['page_id']
    txt_path = TRANSCR_DIR / f'{page_id}.txt'
    if not txt_path.exists():
        continue
    m = re.match(r'B(\d+)', page_id)
    if not m:
        continue
    book = f'B{m.group(1)}'
    content = txt_path.read_text(encoding='utf-8').strip()
    n = len([l for l in content.splitlines() if not l.startswith('===')])
    book_lines[book] += n
    book_pages[book] += 1

print(f"{'Buch':<6} {'Seiten':>7}  {'Zeilen':>8}")
print('-' * 26)
for book in sorted(book_lines):
    print(f"{book:<6} {book_pages[book]:>7}  {book_lines[book]:>8}")
print('-' * 26)
print(f"{'Gesamt':<6} {sum(book_pages.values()):>7}  {sum(book_lines.values()):>8}")

# Vorschau: erste Seite
print()
first_txt = TRANSCR_DIR / f'{pages[0]["page_id"]}.txt'
if first_txt.exists():
    print('--- Vorschau:', first_txt.name, '---')
    print(first_txt.read_text(encoding='utf-8')[:600])

Buch    Seiten    Zeilen
--------------------------
B1         132      3553
B2         164      5137
B3         151      4992
B4         165      5920
B5         155      5701
B6         156      5654
--------------------------
Gesamt     923     30957

--- Vorschau: B1_P012.txt ---
=== Buch 1, Seite 012 ===
ms. germ. quant. 222
Journal of a journey
from London to Plymouth
&. a Voyage
on board his Majesty's Ship
the Resolution
Captain Cook Commander
from Eumouth
to the Cape of good Hope.
from July y.13½November
Ex
(Bibliography. Regie
Berclinen

